# Jordan Statistical Yearbook Agent
An AI agent that answers questions in **Arabic or English** about Jordan's official statistics across 53 datasets. Powered by Google Gemini, LangChain, LangGraph, and Gradio.

**Before running:** Set `GOOGLE_API_KEY` in your `.env` file. Optionally add `LANGCHAIN_API_KEY` for tracing.

**How to run:** Kernel → Restart & Run All. The Gradio UI launches at Cell 9.

---
## Cell 1 — Imports & Setup
Loads all required libraries. `nest_asyncio.apply()` patches Jupyter's event loop so Gradio's async streaming (`astream_events`) can run inside a notebook without raising `RuntimeError: Task got Future attached to a different loop`.

In [7]:
import nest_asyncio
nest_asyncio.apply()          # lets Jupyter's event loop be reused by astream_events

from dotenv import load_dotenv
load_dotenv()

from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langgraph.checkpoint.memory import MemorySaver
import plotly.graph_objects as go
import plotly.io as pio
import sqlite3
import pandas as pd
import numpy as np
import os
from collections import deque

print("Imports OK.")


Imports OK.


## Cell 2 — Constants & Per-Session State
Defines global constants (colors, font, DB path) and the **per-session data store** using Python's `contextvars` module.

Each browser tab gets its own isolated bucket with `df_history`, `figs`, and `prev_temps`. When a tool runs, `_active_sid` tells `_sess()` which bucket to read/write. This prevents data races — without this, two simultaneous users would corrupt each other's chart data.

In [ ]:
import contextvars as _cv
from collections import deque

DB_PATH = "data/syb_database.db"

BLUE    = "#2563eb"
PALETTE = ["#2563eb","#16a34a","#f59e0b","#dc2626","#7c3aed",
           "#0891b2","#db2777","#65a30d","#ea580c","#4f46e5","#0d9488","#b45309"]
FONT    = "Arial, Tahoma, sans-serif"

# Per-session isolated state — keyed by browser session UUID via contextvars
# This prevents data races when multiple users are active simultaneously
_active_sid = _cv.ContextVar("active_sid", default="__default__")
_sessions: dict = {}

def _sess() -> dict:
    sid = _active_sid.get()
    if sid not in _sessions:
        _sessions[sid] = {
            "df_history": deque(maxlen=3),
            "figs":       [],
            "prev_temps": [],
        }
    return _sessions[sid]

print("Constants ready.")


## Cell 3 — Plotly Chart Theme
Registers a custom Plotly template called `"syb"` (Statistical Yearbook) and sets it as the global default. Every chart the agent builds automatically inherits this style — white background, brand color palette, Arabic-friendly font, clean gridlines, and centered titles. The agent only needs to call `fig.update_layout(template="syb")` and gets consistent styling for free.

In [9]:
pio.templates["syb"] = go.layout.Template(
    layout=go.Layout(
        font=dict(family=FONT, size=13, color="#1e293b"),
        plot_bgcolor="white", paper_bgcolor="white", colorway=PALETTE,
        title=dict(x=0.5, xanchor="center",
                   font=dict(size=15, color="#0f172a", family=FONT), pad=dict(b=10)),
        xaxis=dict(showgrid=False, showline=True, linecolor="#e2e8f0", linewidth=1,
                   tickfont=dict(size=11, color="#64748b"),
                   title_font=dict(size=12, color="#475569"),
                   zeroline=False, automargin=True),
        yaxis=dict(showgrid=True, gridcolor="#f1f5f9", gridwidth=1, showline=False,
                   tickfont=dict(size=11, color="#64748b"),
                   title_font=dict(size=12, color="#475569"),
                   zeroline=False, automargin=True, tickformat=",.0f"),
        hoverlabel=dict(bgcolor="white", bordercolor="#cbd5e1",
                        font=dict(size=13, family=FONT, color="#1e293b")),
        margin=dict(l=60, r=50, t=70, b=60),
    )
)
pio.templates.default = "syb"
print("Chart theme registered.")


Chart theme registered.


## Cell 4 — Semantic Table Index (Cached RAG)
Builds the table search index **once at startup**:

1. Finds a working Gemini embedding model (`gemini-embedding-001` → fallbacks)
2. Reads all 53 table descriptions from the metadata table (`sql_table + name_en + name_ar`)
3. Embeds them into a numpy float32 matrix stored in RAM (~650 KB total)

This is **Cached RAG** — no vector database needed. At query time, the user's question is embedded (one API call) and compared to all 53 embeddings via **cosine similarity** in a single numpy matrix multiply. The top-3 most similar tables are returned in milliseconds. A similarity threshold of 0.65 rejects off-topic questions before they waste further API calls.

In [10]:
def _make_embed_model():
    """Try embedding models in order until one works."""
    candidates = [
        "gemini-embedding-001",
        "gemini-embedding-2-preview",
        "models/embedding-001",
    ]
    for name in candidates:
        try:
            m = GoogleGenerativeAIEmbeddings(
                model=name,
                google_api_key=os.getenv("GOOGLE_API_KEY"),
            )
            m.embed_query("test")
            print(f"Embedding model: {name}")
            return m
        except Exception as e:
            print(f"  {name} failed: {e}")
    raise RuntimeError("No embedding model available.")


def _build_table_index(embed_model):
    conn = sqlite3.connect(DB_PATH)
    df_meta = pd.read_sql(
        "SELECT sql_table, name_en, name_ar FROM metadata ORDER BY sql_table", conn
    )
    conn.close()
    texts = [
        f"{r.sql_table}: {r.name_en} | {r.name_ar}" for _, r in df_meta.iterrows()
    ]
    print("Embedding table descriptions (once)...")
    embs = embed_model.embed_documents(texts)
    return texts, np.array(embs, dtype="float32")


_embed_model              = _make_embed_model()
_table_texts, _table_embs = _build_table_index(_embed_model)
print(f"Table index ready — {len(_table_texts)} tables.")


Embedding model: gemini-embedding-001
Embedding table descriptions (once)...
Table index ready — 53 tables.


## Cell 5 — Observability (LangSmith)
Enables LangSmith tracing if `LANGCHAIN_API_KEY` is present in `.env`. When active, every agent run is logged to [smith.langchain.com](https://smith.langchain.com) under the project `jordan-syb-agent` — including tool calls, model inputs/outputs, latency, and token usage per step.

Completely optional — the agent works without it. Useful for debugging agent reasoning and monitoring production usage.

In [12]:
import os

langsmith_key = os.getenv("LANGCHAIN_API_KEY")

if langsmith_key:
    os.environ["LANGCHAIN_TRACING_V2"] = "true"
    os.environ["LANGCHAIN_PROJECT"]    = "jordan-syb-agent"
    try:
        import langsmith
        print("LangSmith tracing active → https://smith.langchain.com")
        print("Project: jordan-syb-agent")
    except ImportError:
        print("LangSmith key found but package missing.")
        print("Run: pip install langsmith")
else:
    print("LangSmith not configured — add LANGCHAIN_API_KEY to your .env file.")
    print("Get a free key at: smith.langchain.com → Settings → API Keys")


LangSmith tracing active → https://smith.langchain.com
Project: jordan-syb-agent


## Cell 6 — Agent Tools & Security
Defines the 5 tools the agent can call, plus all security infrastructure:

| Tool | What it does |
|------|-------------|
| `find_table` | Semantic search over 53 tables; rejects similarity score < 0.65 |
| `get_schema` | Returns column names + sample values from metadata |
| `run_sql` | Executes SELECT on read-only SQLite; strips `--` and `/* */` comments; blocks PRAGMA, ATTACH, WITH |
| `filter_data` | Runs pandas code on the last result inside a sandboxed `exec()` |
| `create_chart` | Builds Plotly figures inside the same sandbox; stores up to 3 per answer |

**Security layers:**
- `_DB_URI` uses `?mode=ro` → SQLite rejects any write at the driver level
- `_SAFE_BUILTINS` → removes `open()`, `__import__`, `os`, `subprocess`, etc. from `exec()` scope
- All tools write to `_sess()` (session-isolated) instead of globals

In [ ]:
import pathlib
import builtins as _builtins_mod
import re as _re

_DB_URI = pathlib.Path(DB_PATH).resolve().as_uri() + "?mode=ro"

_SAFE_BUILTINS = {
    name: getattr(_builtins_mod, name)
    for name in [
        'abs','all','any','bool','dict','divmod','enumerate','filter',
        'float','frozenset','getattr','hasattr','hash','int','isinstance',
        'issubclass','iter','len','list','map','max','min','next','object',
        'pow','print','range','repr','reversed','round','set','slice',
        'sorted','str','sum','tuple','type','zip',
        'True','False','None',
        'Exception','ValueError','TypeError','KeyError','IndexError','AttributeError',
    ]
    if hasattr(_builtins_mod, name)
}


def _validate_sql(query: str) -> str | None:
    stripped = _re.sub(r"--[^\n]*", "", query)
    stripped = _re.sub(r"/\*.*?\*/", "", stripped, flags=_re.DOTALL)
    first = stripped.strip().split()[0].upper() if stripped.strip() else ""
    if first != "SELECT":
        return (
            f"Error: only plain SELECT queries are allowed. "
            f"Got '{first}' — PRAGMA, ATTACH, WITH, and other statements are blocked."
        )
    return None


@tool
def find_table(query: str) -> str:
    """
    Search for relevant statistical tables by topic using semantic similarity.
    Returns the 3 most relevant tables.
    Call ONLY when the topic is NOT in DATA KNOWLEDGE.
    """
    q_emb = np.array(_embed_model.embed_query(query), dtype="float32")
    norms = (np.linalg.norm(_table_embs, axis=1) * np.linalg.norm(q_emb)).clip(1e-9)
    scores = (_table_embs @ q_emb) / norms
    top = np.argsort(scores)[-3:][::-1]
    if scores[top[0]] < 0.65:
        return "No relevant table found for this topic. This agent covers Jordan statistical data only."
    return "\n".join(f"- {_table_texts[i]}" for i in top)


@tool
def get_schema(sql_table: str) -> str:
    """Returns columns and sample values. Call ONLY to verify exact filter values."""
    conn = sqlite3.connect(_DB_URI, uri=True)
    df = pd.read_sql("SELECT columns_info, sample_values FROM metadata WHERE sql_table = ?",
                     conn, params=(sql_table,))
    conn.close()
    if df.empty:
        return f"Table '{sql_table}' not found."
    return f"Columns: {df.iloc[0]['columns_info']}\n\nSample values:\n{df.iloc[0]['sample_values']}"


@tool
def run_sql(query: str) -> str:
    """
    Executes a SELECT query. Result stored for charting and filtering.
    Can be called multiple times for multi-table questions — each result is kept
    in history (df = latest, df_prev = previous, df_old = oldest).
    """
    err = _validate_sql(query)
    if err:
        return err
    try:
        conn = sqlite3.connect(_DB_URI, uri=True)
        df = pd.read_sql(query, conn)
        conn.close()
        if df.empty:
            return "Query returned no results. Try different filter values or a different table."
        _sess()["df_history"].append(df.copy())
        if len(df) > 20:
            return df.head(20).to_string(index=False) + f"\n\n(Showing first 20 of {len(df)} rows)"
        return df.to_string(index=False)
    except Exception as e:
        return f"SQL Error: {str(e)}"


@tool
def filter_data(pandas_code: str) -> str:
    """
    Filter or transform the most recent SQL result WITHOUT new SQL.
    Use for follow-up requests: 'show only Amman', 'top 5 only', 'sort descending'.

    Available: df (most recent result), pd
    Assign result to `result_df`.

    Example:
      result_df = df[df["المحافظة"] == "عمان"].copy()
    """
    s = _sess()
    if not s["df_history"]:
        return "No data. Call run_sql first."
    try:
        local = {"__builtins__": _SAFE_BUILTINS, "df": s["df_history"][-1].copy(), "pd": pd}
        exec(pandas_code, local)
        result_df = local.get("result_df")
        if result_df is None:
            return "Error: assign result to `result_df`."
        if result_df.empty:
            return "Filter returned no rows. Try different criteria."
        s["df_history"].append(result_df.copy())
        return result_df.to_string(index=False)
    except Exception as e:
        return f"Filter error: {e}"


@tool
def create_chart(plotly_code: str) -> str:
    """
    Build a Plotly chart and add it to the display (up to 3 charts per answer).
    Call once per chart. For multi-table comparisons, call twice with different code.

    Available in scope:
      df      — most recent SQL/filter result
      df_prev — second-to-last result (None if not available)
      df_old  — third-to-last result (None if not available)
      go, pd, BLUE, PALETTE, FONT

    Rules:
      - Assign figure to `fig`
      - Always call fig.update_layout(template="syb", height=440, title="...")
      - Use exact column names from df.columns
    """
    s = _sess()
    if not s["df_history"]:
        return "No data. Call run_sql first."
    if len(s["figs"]) >= 3:
        return "Maximum 3 charts per answer reached."
    try:
        dh = s["df_history"]
        local = {
            "__builtins__": _SAFE_BUILTINS,
            "df":      dh[-1].copy(),
            "df_prev": dh[-2].copy() if len(dh) >= 2 else None,
            "df_old":  dh[-3].copy() if len(dh) >= 3 else None,
            "go": go, "pd": pd,
            "BLUE": BLUE, "PALETTE": PALETTE, "FONT": FONT,
        }
        exec(plotly_code, local)
        fig = local.get("fig")
        if fig is None:
            return "Error: no `fig` variable was created."
        s["figs"].append(fig)
        return f"Chart {len(s['figs'])} created."
    except Exception as e:
        return f"Chart error: {e}"


tools = [find_table, get_schema, run_sql, filter_data, create_chart]
print(f"Tools ready: {[t.name for t in tools]}")


## Cell 7 — System Prompt
The fixed instruction block sent to the LLM at the start of every conversation. It defines:

- **Universal column names** — the Arabic column names that appear identically across all 53 tables (`قيمة المؤشر`, `سنة فترة القياس`, etc.)
- **Tool routing rules** — fast path (known table → skip `find_table`), multi-table path, follow-up filter path
- **DATA KNOWLEDGE map** — explicit table-name shortcuts so the agent doesn't waste an API call on `find_table` for common topics like births, GDP, marriages
- **Aggregation patterns** — how to GROUP BY year, governorate, or sex
- **Multi-chart rules** — when and how to call `create_chart` multiple times
- **Answer format** — always reply in the user's language; lead with the direct answer

In [14]:
system_prompt = """You are an expert data analyst for the Jordan Statistical Yearbook (الكتاب الإحصائي السنوي الأردني).
You have 53 statistical tables covering demographics, economy, health, education, transport, crime, and more.

## UNIVERSAL COLUMN NAMES (same in ALL 53 tables)
- Value column  → "قيمة المؤشر"          (always the number to measure/sum/avg)
- Year column   → "سنة فترة القياس"      (always the calendar year, integer)
- Measure name  → "اسم المؤشر" / "Measure Name"
- Governorate   → "المحافظة" / "Governorate"
- Sex           → "الجنس" / "Sex"  (ذكر/Male, أنثى/Female)

## TOOL USAGE — use as few tools as possible
  find_table  → SKIP if topic matches DATA KNOWLEDGE. Call for unknown topics.
  get_schema  → SKIP if columns are already known. Only for verifying filter values.
  run_sql     → Fetch data. Call multiple times for multi-table questions.
  filter_data → For follow-ups on existing data ('show only Amman', 'top 5', 'sort').
  create_chart → After run_sql/filter_data. Call multiple times for multiple charts.

Fast path (known topic):   run_sql → create_chart
Multi-table question:      run_sql (table A) → run_sql (table B) → create_chart (merge or compare)
Follow-up filter:          filter_data → create_chart
Unknown topic:             find_table → run_sql → create_chart

## MULTI-TABLE QUESTIONS
For questions comparing two datasets (e.g. births vs deaths, GDP vs employment):
1. Call run_sql on table A → stored as df
2. Call run_sql on table B → df becomes df_prev, new result is df
3. Inside create_chart, merge: merged = pd.merge(df, df_prev, on="سنة فترة القياس", suffixes=("_b","_a"))
4. Build a comparison chart using the merged DataFrame

## MULTI-CHART ANSWERS
Call create_chart multiple times when the question warrants different views:
- Chart 1: overall trend (line chart by year)
- Chart 2: breakdown (bar chart by governorate or by sex)
- Chart 3: comparison (two datasets overlaid)
Maximum 3 charts per answer.

## AGGREGATION RULES
- Jordan total   → SELECT "سنة فترة القياس", SUM("قيمة المؤشر") AS total FROM "t" GROUP BY "سنة فترة القياس"
- By year        → GROUP BY "سنة فترة القياس" ORDER BY "سنة فترة القياس" ASC
- By governorate → GROUP BY "المحافظة"
- Total / إجمالي → SUM("قيمة المؤشر") | Average / متوسط → AVG("قيمة المؤشر")

## DATA KNOWLEDGE — use these table names directly, no need for find_table
- births / مواليد          → SYB_3_3
- deaths / وفيات            → use find_table("deaths") or find_table("وفيات")
- marriages / زواج          → SYB_3_11
- divorce / طلاق            → SYB_3_17, SYB_3_19v2, SYB_3_20
- GDP / الناتج المحلي      → SYB_23_1, SYB_23_3, SYB_23_6, SYB_23_10
- education / تعليم        → SYB_13_x  |  health / صحة → SYB_14_x
- employment / توظيف       → SYB_18_x  |  crime / جرائم → SYB_17_x
- roads / transport        → SYB_11_x  |  tourism / سياحة → SYB_15_x
- population / سكان        → SYB_18_3, SYB_3_3

## VISUALIZATION
After each run_sql or filter_data, think: does this data deserve a chart?
Choose the best chart type — you have full freedom with go.*.
df = most recent result | df_prev = previous | df_old = oldest (check for None before use).
Always assign to `fig` and call fig.update_layout(template="syb", height=440, title="...").

## RECOVERY
If a query returns no results → retry with different filter values or a different table (max 2 retries).

## ANSWER FORMAT
- Reply in the SAME language the user used.
- Lead with the direct answer. Add insight: trend, highest, lowest, or relationship.
- Keep text concise — charts carry the visual story.
"""

print("System prompt ready.")


System prompt ready.


## Cell 8 — Model Manager & Agent
Builds the LangChain agent and wraps it in `ModelManager` for automatic quota-based model rotation.

**How the agent works:**
- Uses `create_agent` with native Gemini function calling (not ReAct prompting)
- `MemorySaver` stores the full conversation history per `thread_id` (one UUID per browser session)
- Each question = 3–6 LLM calls in a think → call tool → observe → think loop

**Model rotation priority:**
```
gemini-2.0-flash-lite  →  gemini-2.0-flash  →  gemini-2.5-flash  →  gemini-3.1-flash-lite
```
On a 429 quota error, `switch_next()` silently moves to the next model and retries — no restart needed. If all models are exhausted, an error is shown to the user.

In [15]:
MODELS = [
    "gemini-2.0-flash-lite",
    "gemini-2.0-flash",
    "gemini-2.5-flash",
    "gemini-3.1-flash-lite",
]

memory = MemorySaver()

class ModelManager:
    """
    Holds the active LLM + agent pair.
    .switch_next() on quota error moves to the next model with no cell re-run.
    """

    def __init__(self, models, tools, system_prompt, checkpointer):
        self._models       = models
        self._tools        = tools
        self._prompt       = system_prompt
        self._checkpointer = checkpointer
        self._idx          = 0
        self.llm           = None
        self.agent         = None
        self._activate(0)

    def _activate(self, idx):
        name = self._models[idx]
        self.llm = ChatGoogleGenerativeAI(
            model=name,
            google_api_key=os.getenv("GOOGLE_API_KEY"),
            temperature=0,
            streaming=True,
        )
        self.agent = create_agent(
            model=self.llm,
            tools=self._tools,
            system_prompt=self._prompt,
            checkpointer=self._checkpointer,
        )
        self._idx = idx
        print(f"Active model: {name}")

    def switch_next(self):
        """Move to the next model. Returns True if switched, False if list exhausted."""
        if self._idx < len(self._models) - 1:
            self._activate(self._idx + 1)
            return True
        print("All models exhausted.")
        return False

    @property
    def current_name(self):
        return self._models[self._idx]


model_mgr = ModelManager(MODELS, tools, system_prompt, memory)
print("Agent ready.")


Active model: gemini-2.0-flash-lite
Agent ready.


## Cell 9 — Gradio UI
The full chat interface. Every response yields **15 Gradio outputs** simultaneously:

| Output | Description |
|--------|-------------|
| `chatbot` | Streaming word-by-word conversation |
| `plot1/2/3` | Up to 3 interactive Plotly charts, shown/hidden dynamically |
| `token_out` | Active model name displayed after each response |
| `sug1/2/3` | Auto-generated follow-up question buttons |
| `dl_data` | Download latest SQL result as CSV |
| `dl_chart1/2/3` | Download charts as self-contained interactive HTML |

**Security checks inside `respond()` — in order:**
1. Empty input → skip
2. Length > 500 chars → reject with message
3. Prompt injection patterns (12 bilingual regex) → reject
4. Rate limit (10 requests / 60 seconds) → reject with wait time
5. Agent call with streaming

**Session isolation:**
- `demo.load()` assigns a UUID to each browser tab on page load
- Clear button issues a **new UUID** → LangGraph starts a fresh `thread_id` with no memory of the previous conversation
- `_active_sid.set(session_id)` at the start of `respond()` ensures tools write to the correct session bucket

In [ ]:
import gradio as gr
import traceback
import asyncio
import tempfile
import time
import re
import uuid
from collections import deque as _deque

_MAX_RETRIES      = 3
RATE_LIMIT_MAX    = 10
RATE_LIMIT_WINDOW = 60
MAX_INPUT_LEN     = 500

_request_times = _deque()  # global — shared rate limit across all sessions (intentional)

_INJECTION_PATTERNS = [
    r"ignore\s+(all\s+)?(previous|prior|above|your)?\s*(instructions?|rules?|prompt|system)",
    r"forget\s+(all\s+)?(previous|prior|above|your)?\s*(instructions?|rules?|prompt|system)",
    r"disregard\s+(all\s+)?(previous|prior|above)?\s*(instructions?|rules?|prompt|system)",
    r"you\s+are\s+now\s+(a\s+)?(different|new|another)",
    r"act\s+as\s+(a\s+)?(?!data|analyst|assistant\s+for\s+jordan)",
    r"pretend\s+(to\s+be|you\s+are)",
    r"your\s+(new\s+)?(system\s+prompt|instructions?\s+are|role\s+is)",
    r"jailbreak|dan\s+mode|developer\s+mode",
    r"تجاهل.{0,20}(التعليمات|النظام|الأوامر)",
    r"انسَ?.{0,20}(التعليمات|السابقة|الأوامر)",
    r"أنت\s+الآن\s+مساعد\s+مختلف",
    r"دورك\s+الجديد",
]
_INJECTION_RE = re.compile("|".join(_INJECTION_PATTERNS), re.IGNORECASE)


def _check_injection(text: str) -> bool:
    return bool(_INJECTION_RE.search(text))


def extract_text(content):
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        out = []
        for p in content:
            if isinstance(p, dict):
                out.append(p.get("text", ""))
            else:
                out.append(str(p))
        return "".join(out)
    return ""


def _check_rate_limit():
    now = time.time()
    while _request_times and now - _request_times[0] > RATE_LIMIT_WINDOW:
        _request_times.popleft()
    if len(_request_times) >= RATE_LIMIT_MAX:
        wait = int(RATE_LIMIT_WINDOW - (now - _request_times[0])) + 1
        return False, wait
    _request_times.append(now)
    return True, 0


def _save_downloads():
    s = _sess()
    for path in s["prev_temps"]:
        try:
            os.unlink(path)
        except Exception:
            pass
    s["prev_temps"].clear()

    data_path   = None
    chart_paths = [None, None, None]

    if s["df_history"]:
        tmp = tempfile.NamedTemporaryFile(delete=False, suffix=".csv", prefix="syb_data_")
        s["df_history"][-1].to_csv(tmp.name, index=False, encoding="utf-8-sig")
        data_path = tmp.name
        s["prev_temps"].append(tmp.name)

    for i, fig in enumerate(s["figs"][:3]):
        tmp = tempfile.NamedTemporaryFile(delete=False, suffix=".html", prefix=f"syb_chart{i+1}_")
        fig.write_html(tmp.name)
        chart_paths[i] = tmp.name
        s["prev_temps"].append(tmp.name)

    return data_path, chart_paths


async def _generate_suggestions(user_msg: str, ai_response: str) -> list:
    try:
        prompt = (
            f'User asked: "{user_msg}"\n'
            f'Response: "{ai_response[:300]}"\n\n'
            "Suggest 3 short follow-up questions about Jordan statistics "
            "in the SAME language as the user's question. "
            "One per line. No numbers or bullets. Under 12 words each."
        )
        result = await model_mgr.llm.ainvoke(prompt)
        lines = [
            l.strip().lstrip("•-–0123456789.) ")
            for l in result.content.strip().split("\n")
            if l.strip()
        ]
        return (lines + ["", "", ""])[:3]
    except Exception:
        return ["", "", ""]


# Output order (15 total):
# chatbot | plot1 | plot2 | plot3 | token_out | msg_box | error_box
# _sugs | sug1 | sug2 | sug3 | dl_data | dl_chart1 | dl_chart2 | dl_chart3

def _interim(chatbot_val):
    return (
        chatbot_val,
        gr.update(), gr.update(), gr.update(),
        "", "",
        gr.update(visible=False, value=""),
        ["", "", ""],
        gr.update(visible=False),
        gr.update(visible=False),
        gr.update(visible=False),
        gr.update(visible=False),
        gr.update(visible=False),
        gr.update(visible=False),
        gr.update(visible=False),
    )


async def respond(user_msg, history, session_id):
    if not user_msg.strip():
        yield _interim(history)
        return

    if len(user_msg) > MAX_INPUT_LEN:
        msg = f"⚠️ سؤالك طويل جداً ({len(user_msg)} حرف). الحد الأقصى {MAX_INPUT_LEN} حرف. | Message too long ({len(user_msg)} chars). Max is {MAX_INPUT_LEN}."
        yield _interim(history + [{"role": "user", "content": user_msg}, {"role": "assistant", "content": msg}])
        return

    if _check_injection(user_msg):
        msg = "⛔ تم رفض رسالتك. يُرجى طرح سؤال عن الإحصاءات الأردنية. | Message rejected. Please ask a question about Jordan statistics."
        yield _interim(history + [{"role": "user", "content": user_msg}, {"role": "assistant", "content": msg}])
        return

    allowed, wait = _check_rate_limit()
    if not allowed:
        msg = f"⏳ Too many requests. Please wait {wait} seconds before asking again."
        yield _interim(history + [{"role": "user", "content": user_msg}, {"role": "assistant", "content": msg}])
        return

    # Bind this request to its session so tools write to the correct store
    _active_sid.set(session_id)
    _sess()["figs"].clear()

    history = history + [{"role": "user", "content": user_msg}]
    yield _interim(history)

    config  = {"configurable": {"thread_id": session_id}}
    partial = ""
    retries = 0

    try:
        while True:
            try:
                async for event in model_mgr.agent.astream_events(
                    {"messages": [("human", user_msg)]},
                    config=config,
                    version="v2",
                ):
                    if event["event"] == "on_chat_model_stream":
                        chunk = event["data"].get("chunk")
                        if chunk and chunk.content:
                            text = extract_text(chunk.content)
                            if text:
                                partial += text
                                yield _interim(history + [{"role": "assistant", "content": partial}])
                break

            except Exception as e:
                err   = str(e)
                etype = type(e).__name__
                is_quota      = "429" in err or "RESOURCE_EXHAUSTED" in err or "quota" in err.lower()
                is_disconnect = "RemoteProtocolError" in etype or "Server disconnected" in err

                if is_quota and model_mgr.switch_next():
                    partial = ""
                    retries = 0
                    continue

                if is_disconnect and retries < _MAX_RETRIES:
                    retries += 1
                    await asyncio.sleep(2 * retries)
                    partial = ""
                    continue

                raise

        if not partial:
            partial = "لم يتم الحصول على إجابة."

        sugs = await _generate_suggestions(user_msg, partial)

        s    = _sess()
        fig1 = s["figs"][0] if len(s["figs"]) > 0 else None
        fig2 = s["figs"][1] if len(s["figs"]) > 1 else None
        fig3 = s["figs"][2] if len(s["figs"]) > 2 else None

        data_path, chart_paths = _save_downloads()

        yield (
            history + [{"role": "assistant", "content": partial}],
            gr.update(value=fig1, visible=fig1 is not None),
            gr.update(value=fig2, visible=fig2 is not None),
            gr.update(value=fig3, visible=fig3 is not None),
            f"[{model_mgr.current_name}]",
            "",
            gr.update(visible=False, value=""),
            sugs,
            gr.update(value=sugs[0], visible=bool(sugs[0])),
            gr.update(value=sugs[1], visible=bool(sugs[1])),
            gr.update(value=sugs[2], visible=bool(sugs[2])),
            gr.update(value=data_path,      visible=data_path is not None),
            gr.update(value=chart_paths[0], visible=chart_paths[0] is not None),
            gr.update(value=chart_paths[1], visible=chart_paths[1] is not None),
            gr.update(value=chart_paths[2], visible=chart_paths[2] is not None),
        )

    except Exception:
        print(traceback.format_exc())
        user_err = "⚠️ حدث خطأ غير متوقع. يُرجى المحاولة مرة أخرى. | An unexpected error occurred. Please try again."
        yield (
            history,
            gr.update(), gr.update(), gr.update(),
            "", "",
            gr.update(visible=True, value=user_err),
            ["", "", ""],
            gr.update(visible=False), gr.update(visible=False), gr.update(visible=False),
            gr.update(visible=False), gr.update(visible=False),
            gr.update(visible=False), gr.update(visible=False),
        )


def _use_suggestion(idx):
    async def handler(sugs, history, session_id):
        q = sugs[idx] if sugs and len(sugs) > idx else ""
        if not q:
            yield _interim(history)
            return
        async for chunk in respond(q, history, session_id):
            yield chunk
    return handler


with gr.Blocks(title="مساعد الكتاب الإحصائي السنوي") as demo:
    gr.Markdown(
        "# مساعد الكتاب الإحصائي السنوي\n"
        "اسأل عن أي إحصائية من 53 جدول | Ask about any statistic from 53 tables"
    )

    _sugs      = gr.State(["", "", ""])
    session_id = gr.State(None)

    with gr.Row():
        with gr.Column(scale=2):
            chatbot  = gr.Chatbot(height=500, label="المحادثة")
            msg_box  = gr.Textbox(placeholder="اكتب سؤالك هنا...", label="سؤالك", lines=2, max_lines=6)

            with gr.Row():
                send_btn  = gr.Button("إرسال", variant="primary")
                clear_btn = gr.Button("مسح")

            gr.Markdown("**اقتراحات للمتابعة | Follow-up suggestions**")
            with gr.Row():
                sug1 = gr.Button("", visible=False, size="sm", variant="secondary")
                sug2 = gr.Button("", visible=False, size="sm", variant="secondary")
                sug3 = gr.Button("", visible=False, size="sm", variant="secondary")

            gr.Markdown("**تحميل النتائج | Download**")
            with gr.Row():
                dl_data   = gr.DownloadButton("⬇ البيانات CSV",  visible=False, size="sm")
                dl_chart1 = gr.DownloadButton("⬇ الرسم 1 HTML", visible=False, size="sm")
                dl_chart2 = gr.DownloadButton("⬇ الرسم 2 HTML", visible=False, size="sm")
                dl_chart3 = gr.DownloadButton("⬇ الرسم 3 HTML", visible=False, size="sm")

        with gr.Column(scale=1):
            plot1     = gr.Plot(label="الرسم البياني 1")
            plot2     = gr.Plot(label="الرسم البياني 2", visible=False)
            plot3     = gr.Plot(label="الرسم البياني 3", visible=False)
            token_out = gr.Textbox(label="النموذج المستخدم", interactive=False)

    error_box = gr.Textbox(label="Error Details", visible=False, interactive=False, lines=4)

    gr.Examples(
        examples=[
            "كم عدد المواليد الذكور في عمان عام 2020؟",
            "قارن عدد المواليد في كل محافظة عام 2022",
            "العلاقة بين المواليد والوفيات في الأردن",
            "What is the GDP of Jordan in 2023?",
            "اعطني اتجاه الطلاق في الاردن على مر السنين",
        ],
        inputs=msg_box,
    )

    _out = [
        chatbot, plot1, plot2, plot3, token_out, msg_box, error_box, _sugs,
        sug1, sug2, sug3,
        dl_data, dl_chart1, dl_chart2, dl_chart3,
    ]

    demo.load(lambda: str(uuid.uuid4()), outputs=[session_id])

    send_btn.click(respond, [msg_box, chatbot, session_id], _out)
    msg_box.submit(respond, [msg_box, chatbot, session_id], _out)

    # Clear resets the UI AND starts a fresh LangGraph thread (new UUID)
    # so the agent has no memory of the previous conversation
    clear_btn.click(
        lambda: (
            [], None, None, None, "", "",
            gr.update(visible=False, value=""),
            ["", "", ""],
            gr.update(visible=False), gr.update(visible=False), gr.update(visible=False),
            gr.update(visible=False), gr.update(visible=False),
            gr.update(visible=False), gr.update(visible=False),
            str(uuid.uuid4()),
        ),
        outputs=_out + [session_id],
    )

    sug1.click(_use_suggestion(0), [_sugs, chatbot, session_id], _out)
    sug2.click(_use_suggestion(1), [_sugs, chatbot, session_id], _out)
    sug3.click(_use_suggestion(2), [_sugs, chatbot, session_id], _out)

demo.launch()
